# cv_r1 学習環境セットアップ【Colab・deploy の前に1回】

学習側 Drive に **fork clone（= 学習ルート）** を作り、底モデル等を配置する。deploy ノートより先に実行。

| 区分 | 内容 | 永続性 |
|---|---|---|
| **1回だけ**（§1,§3,§4） | fork clone / `initialize.py`（bert・slm・**pretrained_jp_extra**）/ sanity | Drive に永続 |
| **セッション毎**（§2） | `pip install -r requirements.txt` | Colab VM 揮発（毎回実行） |

- 底モデル（warm-start 元）= `pretrained_jp_extra/{G_0, D_0, WD_0}.safetensors`
  （HF `litagin/Style-Bert-VITS2-2.0-base-JP-Extra` から initialize.py が自動取得）。
  **公開 README ではこの底モデルのライセンスに ckpt が従属することを明記**（HF リポジトリのライセンス表記を確認）。
- Drive 容量目安: clone＋bert/slm/pretrained ≈ 数 GB ＋ データセット ≈ 5 GB。無料枠(15GB)は要注意。


In [ ]:
# ===== §1 Drive マウント & fork clone（1回だけ）=====
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import subprocess, os

FORK_URL = 'https://github.com/slp-hu/Style-Bert-VITS2.git'   # 公開 fork（匿名 clone 可）
BRANCH   = 'layer-b-cadence-seq'
PARENT   = Path('/content/drive/MyDrive')           # clone を置く親（任意）
BASE     = PARENT / 'Style-Bert-VITS2'              # 学習ルート = fork clone 直下（deploy の BASE と一致させる）

if not BASE.exists():
    print('clone:', FORK_URL, '->', BASE)
    subprocess.run(['git', 'clone', '-b', BRANCH, FORK_URL, str(BASE)], check=True)
else:
    print('既存 clone を使用:', BASE)

os.chdir(BASE)
br = subprocess.run(['git','rev-parse','--abbrev-ref','HEAD'], capture_output=True, text=True).stdout.strip()
print('branch:', br)
assert br == BRANCH, f"★branch が {BRANCH} でない: {br} → git checkout {BRANCH}"


In [ ]:
# ===== §2 依存インストール（セッション毎に実行）=====
# 注意: Colab 標準の torch を requirements が下書き換えする場合がある。
# 学習セッションで確立済みの手順（07a 系ノートの流儀）があればそちらを正とする。
import subprocess, sys
r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'],
                   capture_output=True, text=True)
print(r.stdout[-2000:] or '(quiet)')
if r.returncode != 0:
    print(r.stderr[-3000:]); raise SystemExit('pip 失敗')
import torch
print('torch:', torch.__version__, '| cuda:', torch.cuda.is_available())


In [ ]:
# ===== §3 initialize（1回だけ・Drive に永続）=====
# bert / slm(wavlm) / pretrained / pretrained_jp_extra(底モデル) を clone 内へダウンロード
import subprocess, sys
r = subprocess.run([sys.executable, 'initialize.py', '--skip_default_models'],
                   capture_output=True, text=True)
print(r.stdout[-2000:]); print(r.stderr[-1000:])
assert r.returncode == 0, 'initialize.py 失敗'


In [ ]:
# ===== §4 sanity（毎回実行して良い）=====
import hashlib
from pathlib import Path

need = [
    'pretrained_jp_extra/G_0.safetensors',   # 底モデル（warm-start 元）
    'pretrained_jp_extra/D_0.safetensors',
    'pretrained_jp_extra/WD_0.safetensors',
    'slm/wavlm-base-plus/pytorch_model.bin',
    'configs/config_jp_extra.json',
    'configs/paths.yml',
]
ng = 0
for f in need:
    p = BASE / f
    ok = p.exists()
    print(('OK ' if ok else '★NG') , f, f'({p.stat().st_size/2**20:.0f} MiB)' if ok else '')
    ng += (not ok)
assert ng == 0, '不足あり: initialize.py の出力を確認'

h = hashlib.sha256((BASE/'configs'/'config_jp_extra.json').read_bytes()).hexdigest()
print('config_jp_extra.json sha256 一致:', h == '69d907db9f38bf05f52d117814d38808635a4975de38fae5c775f4ff47d42187')

import yaml
py = yaml.safe_load(open(BASE/'configs'/'paths.yml', encoding='utf-8'))
print('paths.yml dataset_root:', py.get('dataset_root'), '（Data のままで良い: バンドルは Data/cv_r1 に展開される）')


## 次の手順
1. **`cv_r1_deploy_colab.ipynb`** を実行（`BASE` を本ノートと同じ値に）→ §3 検証ゲート全通過
2. **bert_gen / style_gen**（このセッションのまま実行可。★コマンドは fork の README/流儀に合わせて確認）:
   ```
   python bert_gen.py  -c Data/cv_r1/config.json
   python style_gen.py -c Data/cv_r1/config.json
   ```
   ※ `preprocess_text` は**走らせない**（esd 配置済み・spk2id 再生成の恐れ）。
3. **学習**: `python train_ms_jp_extra.py -c Data/cv_r1/config.json -m cv_r1`（batch=16 / 10 epoch ≈ 8,900 step, freeze_decoder=True は config に設定済み。★引数の流儀は fork に合わせて確認）
